In [1]:
import pandas as pd
from arcgis.features import FeatureLayer

In [2]:
def categorize_values(census_df, categories_df, category_column, grouping_prefix_column, variable_category_column):
    """
    Categorizes and groups the values in the census DataFrame according to the specified categories DataFrame.

    Parameters:
    census_df (DataFrame): DataFrame containing census data.
    categories_df (DataFrame): DataFrame containing the mapping of variable codes to categories.
    category_column (str): Name of the column in categories_df that contains the category labels.
    grouping_prefix (str): Prefix to add to grouped variable codes and dataset/category names.

    Returns:
    DataFrame: Grouped and categorized DataFrame.
    """
    census_df_copy = census_df.copy()
    census_df_copy['value'] = census_df_copy['value'].astype(float)
    census_df_copy.drop(columns=['variable_category'], inplace=True)
    
    joined_data = census_df_copy.merge(categories_df, on='variable_code', how='left')
    joined_data.sort_values(by='variable_code', inplace=True)
    
    # Identify grouping columns excluding certain fixed columns
    group_columns = [column for column in census_df_copy if column not in ['value', 'variable_code', 'variable_name', 'MarginOfError','variable_category', 'OBJECTID']]
    group_columns.append(category_column)
    group_columns.append(variable_category_column)
    group_columns.append(grouping_prefix_column)
    
    grouped_data = joined_data.groupby(group_columns, as_index=False, dropna=False).agg({
        'value': 'sum',
        'variable_code': lambda x: ', '.join(x)
    })
    grouped_data['variable_code'] = grouped_data[grouping_prefix_column] +": " + grouped_data['variable_code']
    # Insert missing columns to maintain structure
    var_code_col_location = census_df.columns.get_loc('variable_code')
    var_name_col_location = census_df.columns.get_loc('variable_name')
    var_moe_col_location = census_df.columns.get_loc('MarginOfError')
    
    grouped_data.insert(var_moe_col_location, 'MarginOfError', '')
    grouped_data.insert(var_name_col_location, 'variable_name', '')
    
    grouped_data['variable_name'] = grouped_data[category_column]
    grouped_data['dataset'] = grouped_data[grouping_prefix_column] + ": " + grouped_data['dataset']
    columns_to_keep = [column for column in census_df if column not in ['OBJECTID']]
    grouped_data = grouped_data[columns_to_keep]
    
    return grouped_data
#Helper function that is used to concatenate census data return
def create_or_append_df(df, summary_df):
    if df.empty:
        df = summary_df.copy()
    else:
        df = pd.concat([df, summary_df])
    return df

def sum_multiple_variables(df, variable_list):
    df_values=pd.DataFrame()
    for variable,variable_category in variable_list:
        print(variable)
        summed_df = sum_across_levels(df,variable, variable_category)
        
        df_values = create_or_append_df(df_values, summed_df)
    return df_values

def sum_across_levels(df, variable_code, category_name):
    """
    Sums values across multiple geographic and sample levels for a specific variable code.

    Parameters:
    df (DataFrame): DataFrame containing the data to be aggregated.
    variable_code (str): The variable code to filter and aggregate.
    category_name (str): The category name to be associated with the aggregated data.

    Returns:
    DataFrame: Combined summary DataFrame with summed values for each geographic level.
    """
    filtered_df = df.loc[df['variable_code'] == variable_code]
    
    
    basin_summary = filtered_df.groupby([
        'dataset', 'sample_level', 'variable_name', 'variable_code', 
        'variable_category', 'year_sample'
    ], as_index=False).sum(['value'])
    
    county_summary = filtered_df.groupby([
        'dataset', 'sample_level', 'variable_name', 'variable_code', 
        'variable_category', 'year_sample', 'county_name'
    ], as_index=False).sum(['value'])
    
    north_south_summary = filtered_df.groupby([
        'dataset', 'sample_level', 'variable_name', 'variable_code', 
        'variable_category', 'year_sample', 'north_south'
    ], as_index=False).sum(['value'])
    
    state_summary = filtered_df.groupby([
        'dataset', 'sample_level', 'variable_name', 'variable_code', 
        'variable_category', 'year_sample', 'state_name'
    ], as_index=False).sum(['value'])
    
    basin_summary['Geography'] = 'Basin'
    county_summary['Geography'] = county_summary['county_name']
    north_south_summary['Geography'] = north_south_summary['north_south']
    state_summary['Geography'] = state_summary['state_name']
    
    columns_to_keep = ['variable_code', 'variable_name', 'variable_category', 'value', 'Geography', 'year_sample', 'dataset', 'sample_level']
    
    basin_summary = basin_summary[columns_to_keep]
    county_summary = county_summary[columns_to_keep]
    north_south_summary = north_south_summary[columns_to_keep]
    state_summary = state_summary[columns_to_keep]
    
    combined_summary = pd.concat([basin_summary, county_summary, north_south_summary, state_summary], ignore_index=True)
    
    combined_summary['Category'] = combined_summary['variable_category']
    combined_summary.drop(columns=['variable_category'], inplace=True)
    
    return combined_summary


In [3]:
county_lookup = {
    '005': 'Douglas County (Tahoe Basin)',
    '017': 'El Dorado County (Tahoe Basin)',
    '031': 'Washoe County (Tahoe Basin)',
    '061': 'Placer County (Tahoe Basin)'
}

state_lookup = {
    '32': 'Nevada',
    '06': 'California'
}

inflation_adjustment = {
    '2000': 1.57,
    '2010': 1.24}

In [5]:
additional_2023 = pd.read_csv('combined_csv.csv')

In [13]:
all_data_clean = additional_2023.loc[(additional_2023['county']!='510')&(additional_2023['value']>=0)]
all_data_clean.loc[all_data_clean['county'].isin(['005','017']),'north_south'] = 'South Lake'
all_data_clean.loc[all_data_clean['county'].isin(['031','061']),'north_south'] = 'North Lake'
all_data_clean['county_name'] = all_data_clean['county'].apply(lambda x: county_lookup.get(x, None))
all_data_clean['state_name'] = all_data_clean['state'].apply(lambda x: state_lookup.get(x, None))

In [14]:
combined_csv = all_data_clean.copy()

In [15]:
def add_leading_zero_based_on_condition(row):
    if row['sample_level'] == 'tract' and len(row['TRPAID']) == 14:
        return row['TRPAID'].zfill(15)
    elif row['sample_level'] == 'block group' and len(row['TRPAID']) == 15:
        return row['TRPAID'].zfill(16)
    else:
        return row['TRPAID']
combined_csv['GEO_ID'] = combined_csv['GEO_ID'].astype(str).str.split('.').str[0]
combined_csv['census_geom_year'] = combined_csv['census_geom_year'].astype(int)
combined_csv['TRPAID'] = combined_csv['GEO_ID'].astype(str) + combined_csv['census_geom_year'].astype(str)
combined_csv['TRPAID'] = combined_csv.apply(add_leading_zero_based_on_condition, axis=1)
combined_csv
combined_csv['state'] = combined_csv['state'].astype(str).str.zfill(2)
combined_csv['county'] = combined_csv['county'].astype(str).str.zfill(3)

In [11]:
variable_list = combined_csv[['variable_code', 'variable_category']].drop_duplicates().values.tolist()

In [16]:
summed_df = sum_multiple_variables(combined_csv, variable_list)

B25004_002E
B25004_003E
B25004_004E
B25004_005E
B25004_007E


In [17]:
summed_df.to_csv('summed_df.csv', index=False)

In [4]:
service_url = 'https://maps.trpa.org/server/rest/services/Demographics/MapServer/28'

feature_layer = FeatureLayer(service_url)
query_result = feature_layer.query()
# Convert the query result to a list of dictionaries
feature_list = query_result.features

# Create a pandas DataFrame from the list of dictionaries
all_data = pd.DataFrame([feature.attributes for feature in feature_list])

all_data_clean = all_data.loc[(all_data['county']!='510')&(all_data['value']>=0)]
all_data_clean.loc[all_data_clean['county'].isin(['005','017']),'north_south'] = 'South Lake'
all_data_clean.loc[all_data_clean['county'].isin(['031','061']),'north_south'] = 'North Lake'
all_data_clean['county_name'] = all_data_clean['county'].apply(lambda x: county_lookup.get(x, None))
all_data_clean['state_name'] = all_data_clean['state'].apply(lambda x: state_lookup.get(x, None))


all_data_clean = all_data_clean.dropna(subset='value')


C:\Users\amcclary\AppData\Local\Temp\ipykernel_36464\949055721.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_data_clean.loc[all_data_clean['county'].isin(['005','017']),'north_south'] = 'South Lake'
C:\Users\amcclary\AppData\Local\Temp\ipykernel_36464\949055721.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_data_clean['county_name'] = all_data_clean['county'].apply(lambda x: county_lookup.get(x, None))
C:\Users\amcclary\AppData\Local\Temp\ipykernel_36464\949055721.py:15: SettingWithCopy

In [22]:
grouping_variables = pd.read_csv('Lookup_Lists\Overall_Grouping_List.csv')

In [21]:
data_2023 = all_data_clean.loc[all_data_clean['year_sample'] == 2023]

In [23]:
variables_2023 = data_2023[['variable_code', 'variable_category']].drop_duplicates()

In [43]:
categorized_census_data = categorize_values(data_2023, grouping_variables, 'broad_category', 'grouping_prefix', 'variable_category')

In [44]:
categorized_census_data= categorized_census_data.loc[categorized_census_data['variable_category'].isna() == False]

In [45]:
categorized_census_data.to_csv('Categorized_Census_Data.csv', index=False)

# Append data to geodatabase

In [3]:
categorized_census_data = pd.read_csv('Categorized_Census_Data.csv')

In [10]:
combined_csv = categorized_census_data.copy()
#add a leading 0 to the TRPAID column if the length is 15
def add_leading_zero_based_on_condition(row):
    if row['sample_level'] == 'tract' and len(row['TRPAID']) == 14:
        return row['TRPAID'].zfill(15)
    elif row['sample_level'] == 'block group' and len(row['TRPAID']) == 15:
        return row['TRPAID'].zfill(16)
    else:
        return row['TRPAID']
combined_csv['GEO_ID'] = combined_csv['GEO_ID'].astype(str).str.split('.').str[0]
combined_csv['census_geom_year'] = combined_csv['census_geom_year'].astype(int)
combined_csv['TRPAID'] = combined_csv['GEO_ID'].astype(str) + combined_csv['census_geom_year'].astype(str)
combined_csv['TRPAID'] = combined_csv.apply(add_leading_zero_based_on_condition, axis=1)
combined_csv
combined_csv['state'] = combined_csv['state'].astype(str).str.zfill(2)
combined_csv['county'] = combined_csv['county'].astype(str).str.zfill(3)
# Drop rows with a missing value
combined_csv = combined_csv.dropna(subset=['value'])
combined_csv.to_csv('combined_csv.csv', index=False)

In [11]:
clean_csv = combined_csv.copy()

# Convert all columns of type 'object' to 'string'
clean_csv = clean_csv.astype({col: 'string' for col in clean_csv.select_dtypes(include='object').columns})

clean_csv.info()
clean_csv['NEIGHBORHOOD'] = clean_csv['NEIGHBORHOOD'].fillna("")
# make sure that all strings are less than 255
clean_csv['variable_code'] = clean_csv['variable_code'].str[:255]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3256 entries, 0 to 3255
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   GEO_ID             3256 non-null   string 
 1   value              3256 non-null   int64  
 2   MarginOfError      0 non-null      float64
 3   state              3256 non-null   string 
 4   county             3256 non-null   string 
 5   tract              3256 non-null   int64  
 6   block_group        2233 non-null   float64
 7   variable_code      3256 non-null   string 
 8   variable_name      3256 non-null   string 
 9   variable_category  3256 non-null   string 
 10  year_sample        3256 non-null   int64  
 11  sample_level       3256 non-null   string 
 12  dataset            3256 non-null   string 
 13  census_geom_year   3256 non-null   int32  
 14  TRPAID             3256 non-null   string 
 15  NEIGHBORHOOD       1023 non-null   string 
 16  Data_Scale         0 non

In [12]:
import arcpy
# Map pandas dtypes to ArcGIS field types
clean_csv['GEO_ID'] = clean_csv['GEO_ID'].astype(str)
type_mapping = {
    'int64': 'LONG',
    'float64': 'DOUBLE',
    'object': 'TEXT',
    'string': 'TEXT',
    'datetime64[ns]': 'DATE'
}

# Set up geodatabase and output table name
gdb_path = r"F:\GIS\PROJECTS\ResearchAnalysis\Demographics\Workspace.gdb"
output_table = "acs_2023_grouped"
output_path = f"{gdb_path}\\{output_table}"
if arcpy.Exists(output_path):
    arcpy.management.Delete(output_path)
    print(f"Deleted existing table: {output_table}")
# Create the table in the geodatabase
arcpy.management.CreateTable(gdb_path, output_table)

# Add fields based on DataFrame dtypes
for col_name, dtype in clean_csv.dtypes.items():
    arcgis_type = type_mapping.get(str(dtype), 'TEXT')  # Default to TEXT if dtype is unknown
    if arcgis_type == 'TEXT':
        arcpy.management.AddField(output_path, col_name, arcgis_type, field_length=255)
    else:
        arcpy.management.AddField(output_path, col_name, arcgis_type)

# Insert data into the table
with arcpy.da.InsertCursor(output_path, clean_csv.columns.tolist()) as cursor:
    for _, row in clean_csv.iterrows():
        cursor.insertRow(row.tolist())

print(f"Table '{output_table}' created and populated in {gdb_path}")

Table 'acs_2023_grouped' created and populated in F:\GIS\PROJECTS\ResearchAnalysis\Demographics\Workspace.gdb


In [14]:
clean_csv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3256 entries, 0 to 3255
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   GEO_ID             3256 non-null   object 
 1   value              3256 non-null   int64  
 2   MarginOfError      0 non-null      float64
 3   state              3256 non-null   string 
 4   county             3256 non-null   string 
 5   tract              3256 non-null   int64  
 6   block_group        2233 non-null   float64
 7   variable_code      3256 non-null   string 
 8   variable_name      3256 non-null   string 
 9   variable_category  3256 non-null   string 
 10  year_sample        3256 non-null   int64  
 11  sample_level       3256 non-null   string 
 12  dataset            3256 non-null   string 
 13  census_geom_year   3256 non-null   int32  
 14  TRPAID             3256 non-null   string 
 15  NEIGHBORHOOD       3256 non-null   string 
 16  Data_Scale         0 non

In [25]:
variables_2023_list = [(row.variable_code, row.variable_category) for row in variables_2023.itertuples(index=False)]

#geography_grouped = sum_multiple_variables

In [28]:
summed_variables = sum_multiple_variables(data_2023, variables_2023_list)

B16004_018E
B16004_062E
B16004_037E
B16004_015E
B16004_059E
B16004_038E
B16004_016E
B16004_060E
B16004_039E
B16004_017E
B16004_061E
B16004_025E
B16004_003E
B16004_047E
B16004_035E
B16004_013E
B16004_057E
B16004_034E
B16004_012E
B16004_056E
B25040_003E
B25040_004E
B16004_032E
B16004_010E
B16004_054E
B16004_033E
B16004_011E
B16004_055E
B16004_045E
B16004_023E
B16004_067E
B16004_044E
B16004_022E
B16004_066E
B16004_042E
B16004_020E
B16004_064E
B16004_043E
B16004_021E
B16004_065E
B16004_030E
B16004_008E
B25040_005E
B25040_006E
B16004_052E
B16004_029E
B16004_007E
B16004_051E
B16004_027E
B16004_005E
B16004_049E
B16004_028E
B16004_006E
B16004_050E
B16004_026E
B16004_004E
B16004_048E
B19052_001E
B08119_019E
B08119_010E
B08119_028E
B08119_046E
B08119_037E
B08119_055E
B25040_007E
B25040_008E
B08008_001E
B23024_001E
B19001_001E
B25044_001E
B25001_001E
B25002_002E
B25002_003E
B25044_004E
B25044_005E
B25044_006E
B25044_007E
B25044_008E
B25044_003E
B25044_002E
B03002_012E
B03002_005E
B03002_006E
B030

In [29]:
summed_variables.to_csv('Summed_Variables_2023.csv', index=False)